# Cheap Prediction Model

In [ ]:
from imbalance_sa.paths import PROJECT_ROOT
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
isem_niv_pimb = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "niv" / "isem_niv_and_pimb_actual.csv",
    index_col="time_UTC", parse_dates=True,
)
# Raw export has some rows out of order and exact duplicate timestamps (Nov 2023)
isem_niv_pimb = isem_niv_pimb[~isem_niv_pimb.index.duplicated(keep="first")].sort_index()

isem_wind_generation = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "wind" / "isem_wind_actual.csv",
    index_col="time_UTC", parse_dates=True,
)

isem_niv = isem_niv_pimb[['NIV (MW)']].copy()
isem_pimb = isem_niv_pimb[['PIMB (€)']].copy()

In [ ]:
from plotly.subplots import make_subplots

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=7)
niv_week = isem_niv.loc[start:end]
pimb_week = isem_pimb.loc[start:end]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=niv_week.index, y=niv_week["NIV (MW)"], name="NIV (MW)"), secondary_y=False)
fig.add_trace(go.Scatter(x=pimb_week.index, y=pimb_week["PIMB (€)"], name="PIMB (€)", line=dict(color="red")), secondary_y=True)

fig.update_yaxes(title_text="NIV (MW)", secondary_y=False)
fig.update_yaxes(title_text="PIMB (€)", secondary_y=True)
fig.update_xaxes(title_text="Date")
fig.update_layout(title_text=f"ISEM NIV and PIMB from {start.date()} to {end.date()}", legend=dict(x=0.01, y=0.99))
fig.show()

In [ ]:
niv_pimb_correlation = isem_niv_pimb.corr().loc["NIV (MW)", "PIMB (€)"]
display(niv_pimb_correlation)